In [ ]:
from Model import fNIRS_T
from Metrics import metrics, load_seed
from Dataloader_EF import read_ef_excel_si as si_data

import torch
import os
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score


class LabelSmoothing(torch.nn.Module):
	def __init__(self, smoothing=0.0):
		super(LabelSmoothing, self).__init__()
		self.confidence = 1.0 - smoothing
		self.smoothing = smoothing

	def forward(self, x, target):
		logprobs = torch.nn.functional.log_softmax(x, dim=-1)
		nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
		nll_loss = nll_loss.squeeze(1)
		smooth_loss = -logprobs.mean(dim=-1)
		loss = self.confidence * nll_loss + self.smoothing * smooth_loss
		return loss.mean()

class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.flooding_level = config['flooding_level']

		if config['mode'] == 0 or config['mode'] == 1:
			self.model = fNIRS_T(n_class=2, sampling_point=200, dim=64, depth=6, heads=8, mlp_dim=64).to(self.device)
		elif config['mode'] == 2:
			self.model = fNIRS_T(n_class=2, sampling_point=100, dim=64, depth=6, heads=8, mlp_dim=64).to(self.device)
		self.criterion = LabelSmoothing(0.1)
		self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['learning_rate'])
		self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
			self.optimizer,
			mode='min',
			factor=0.8,
			patience=5,
			verbose=True,
			min_lr=0.1*config['learning_rate'],
		)

		os.makedirs('Results', exist_ok=True)
	
	def evaluate_epoch(self, subject, eval_loader):
		self.model.load_state_dict(torch.load(f'Results/{subject}.pth'))
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_nirs, eval_labels in eval_loader:
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
				eval_labels = eval_labels.to(self.device)
				
				outputs = self.model(eval_nirs)
				loss = self.criterion(outputs, eval_labels.long())
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				total_loss += loss.item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		self.scheduler.step(loss)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds, zero_division=0)
		f1 = f1_score(all_labels, all_preds, zero_division=0)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		return loss, acc, precision, recall, f1, kappa
	
	def test_subject(self, subject, mode):
		test_dataset, eval_dataset = si_data(subject, mode)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(subject, eval_loader)
		print(f"Subject {subject} | Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")

		return eval_acc, precision, recall, f1, kappa

# Hyperparameters
config = {
	'num_classes': 2,
	'flooding_level': [0.45, 0.40, 0.35],
	'mi_root': '../../Dataset/EF-MI-MA/MI/',
	'ma_root': '../../Dataset/EF-MI-MA/MA/',
	'wg_root' : '../../Dataset/EF-WG/WG/',
	'batch_size': 128,
	'num_epochs': 120,
	'learning_rate': 1e-3,
	'mode': 1,
	'log_name': '3111',
}

# Initialize and run trainer
for run in range(1):
	mode = config['mode']
	num_subject = (29 if mode == 0 or mode == 1 else 26) # MI = 0, MA = 1, WG = 2
	seeds = load_seed()
	acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], [], []
	for subject in range(1, num_subject+1):
		seed = seeds[subject-1]
		# seed = random.randint(1, 2025)
		torch.manual_seed(seed)
		torch.cuda.manual_seed_all(seed)
		print(f"\n=== Subject {subject} ===")
		print(f'Seed is {seed}\n')
		trainer = Trainer(config)
		eval_acc, precision, recall, f1, kappa = trainer.test_subject(subject, mode)
		acc_list.append(eval_acc)
		precision_list.append(precision)
		recall_list.append(recall)
		f1_list.append(f1)
		kappa_list.append(kappa)
	
	metrics(config['log_name'], acc_list, precision_list, recall_list, f1_list, kappa_list)	